In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q -U transformers datasets evaluate scikit-learn huggingface-hub adapters biopython umap-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.5/123.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os

# Change working directory to your Google Drive project folder
project_path = "/content/drive/MyDrive/bio_esm2_project"
os.chdir(project_path)

print("Current Working Directory:", os.getcwd())
# Verify project subdirectories
print("Folder contents:", os.listdir())

Current Working Directory: /content/drive/MyDrive/bio_esm2_project
Folder contents: ['data', 'src', 'models', 'results', 'esm2_tem1_results', 'esm2_tem1_fixed', 'esm2_tem1_phase1', 'best_esm2_tem1_model.pt', 'esm2_tem1_test_predictions.csv']


In [4]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, EsmModel, get_cosine_schedule_with_warmup
from sklearn.metrics import matthews_corrcoef, f1_score, roc_auc_score, accuracy_score, r2_score

# Verify GPU availability
print("--- ENVIRONMENT CHECK ---")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

--- ENVIRONMENT CHECK ---
PyTorch Version: 2.11.0+cpu
CUDA Available:  False


In [5]:
WT_TEM1 = (
    "MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIELDLNSGKILESFRPEERFPMMSTFKVLLCG"
    "AVLSRIDAGQEQLGRRIHYSQNDLVEYSPVTEKHLTDGMTVRELCSAAITMSDNTAANLLLTTIGGPKELTAFLHN"
    "MGDHVTRLDRWEPELNEAIPNDERDTTMPVAMATTLRKLLTGELLTLASRQQLIDWMEADKVAGPLLRSALPAGWF"
    "IADKSGAGERGSRGIIAALGPDGKPSRIVVIYTTGSQATMDERNRQIAEIGASLIKHW"
)

# 3-Letter to 1-Letter Amino Acid Dictionary
AA_MAP = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
    'Gln': 'Q', 'Glu': 'E', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
    'Leu': 'L', 'Phe': 'F', 'Pro': 'P', 'Ser': 'S', 'Thr': 'T',
    'Trp': 'W', 'Tyr': 'Y', 'Val': 'V'
}

def mutate_sequence(hgvs_str, wt_seq):
    """Parses single amino acid point mutations (e.g., 'p.Glu104Lys') and mutates wt_seq."""
    if pd.isna(hgvs_str) or hgvs_str in ['_wt', '_synonymous', 'p.=', '=']:
        return wt_seq

    # Match standard point mutation pattern: p.RefPosAlt (e.g., p.Glu104Lys)
    match = re.match(r"p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})", str(hgvs_str))
    if not match:
        return None  # Skip indels or unparseable non-point mutations for benchmarking

    ref_aa_3, pos_str, alt_aa_3 = match.groups()
    pos = int(pos_str) - 1  # Convert to 0-based index

    ref_aa = AA_MAP.get(ref_aa_3)
    alt_aa = AA_MAP.get(alt_aa_3)

    if ref_aa is None or alt_aa is None or pos >= len(wt_seq):
        return None

    # Verify wild-type residue at position matches
    if wt_seq[pos] != ref_aa:
        return None

    # Apply single point mutation
    seq_list = list(wt_seq)
    seq_list[pos] = alt_aa
    return "".join(seq_list)

In [6]:
import re

# 1. Load the raw MaveDB dataset
raw_path = "data/mavedb_tem1.csv"
df_raw = pd.read_csv(raw_path)

# Drop entries missing essential mutation or score data
df_clean = df_raw.dropna(subset=['score', 'hgvs_pro']).copy()

# 2. Reconstruct mutated full-length sequences using the mutate_sequence function
df_clean['sequence'] = df_clean['hgvs_pro'].apply(lambda x: mutate_sequence(x, WT_TEM1))

# Drop any entries that failed mutation parsing (e.g., out-of-bounds or stop-codon mutations)
df_final = df_clean.dropna(subset=['sequence']).copy()

# 3. Assign binary labels based on the 0.5 fitness score threshold:
# Score >= 0.5 -> Label 1 (Resistant / Functional)
# Score < 0.5  -> Label 0 (Susceptible / Impaired)
THRESHOLD = 0.5
df_final['label'] = (df_final['score'] >= THRESHOLD).astype(int)

# 4. Filter columns required for ESM-2 fine-tuning
df_dataset = df_final[['sequence', 'label', 'score']].reset_index(drop=True)

# 5. Save the processed dataset
output_path = "data/processed_tem1_dataset.csv"
df_dataset.to_csv(output_path, index=False)

# Verification Output
print("--- PROCESSED DATASET GENERATION COMPLETE ---")
print(f"Saved dataset to: {output_path}")
print(f"Total valid single-point mutation sequences: {len(df_dataset)}")
print("Class Distribution (0 = Susceptible, 1 = Resistant):")
print(df_dataset['label'].value_counts().to_dict())
print(f"Proportion Resistant: {df_dataset['label'].mean():.2%}")
print("\nFirst 3 parsed entries:")
print(df_dataset.head(3))

--- PROCESSED DATASET GENERATION COMPLETE ---
Saved dataset to: data/processed_tem1_dataset.csv
Total valid single-point mutation sequences: 4562
Class Distribution (0 = Susceptible, 1 = Resistant):
{0: 2351, 1: 2211}
Proportion Resistant: 48.47%

First 3 parsed entries:
                                            sequence  label     score
0  MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIE...      0  0.391423
1  MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIE...      0  0.005659
2  MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIE...      0  0.003689


In [7]:
data_path = "data/processed_tem1_dataset.csv"
df = pd.read_csv(data_path)

print(f"Loaded dataset with {len(df)} samples.")

Loaded dataset with 4562 samples.


In [9]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate shuffled indices
shuffled_indices = np.random.permutation(len(df))
test_set_size = int(len(df) * 0.20)

test_indices = shuffled_indices[:test_set_size]
train_indices = shuffled_indices[test_set_size:]

train_df = df.iloc[train_indices].reset_index(drop=True)
test_df = df.iloc[test_indices].reset_index(drop=True)

print(f"Train samples: {len(train_df)} | Test samples: {len(test_df)}")

Train samples: 3650 | Test samples: 912


In [11]:
# Define Compute Metrics with MCC, F1, and ROC-AUC
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Probabilities for class 1 (Resistant) needed for ROC-AUC
    probabilities = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

    # Calculate requested metrics
    mcc = matthews_corrcoef(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    auc = roc_auc_score(labels, probabilities)
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "mcc": mcc,
        "f1": f1,
        "roc_auc": auc
    }

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} ({torch.cuda.get_device_name(0)})")

In [14]:
MODEL_CHECKPOINT = "facebook/esm2_t12_35M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [15]:
# 1. Dataset Class for Continuous Fitness Regression
class ProteinRegressionDataset(Dataset):
    def __init__(self, sequences, fitness_scores, tokenizer, max_length=300):
        self.sequences = sequences.tolist() if isinstance(sequences, pd.Series) else sequences
        self.scores = fitness_scores.tolist() if isinstance(fitness_scores, pd.Series) else fitness_scores
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.sequences[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["fitness"] = torch.tensor(self.scores[idx], dtype=torch.float32)
        return item

# 2. ESM-2 Regression Model Architecture

class ESM2FitnessRegressor(nn.Module):
    def __init__(self, checkpoint=MODEL_CHECKPOINT):
        super().__init__()
        self.esm = EsmModel.from_pretrained(checkpoint)
        hidden_size = self.esm.config.hidden_size

        # Attention Pooling Head to isolate variant site
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        # Continuous Fitness Regressor (Outputs a single scalar value)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.esm(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state  # (batch, seq_len, hidden_dim)

        # Attention weights with FP16-safe masking
        attn_weights = self.attention(last_hidden)
        mask = (attention_mask.unsqueeze(-1) == 0)
        attn_weights = attn_weights.masked_fill(mask, -1e4)
        attn_weights = torch.softmax(attn_weights, dim=1)

        context_vector = torch.sum(attn_weights * last_hidden, dim=1)
        score_pred = self.regressor(context_vector).squeeze(-1)  # Output shape: (batch_size,)
        return score_pred

In [16]:
# Load Datasets (Ensure train_df['fitness'] contains continuous MaveDB scores)
train_dataset = ProteinRegressionDataset(train_df['sequence'], train_df['score'], tokenizer)
test_dataset = ProteinRegressionDataset(test_df['sequence'], test_df['score'], tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model = ESM2FitnessRegressor().to(device)

for param in model.parameters():
    param.requires_grad = True

# 3. Optimization Setup
optimizer = torch.optim.AdamW([
    {"params": model.esm.parameters(), "lr": 2e-5},
    {"params": model.attention.parameters(), "lr": 2e-4},
    {"params": model.regressor.parameters(), "lr": 1e-3}
], weight_decay=0.01)

epochs = 15
total_steps = len(train_loader) * epochs
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

criterion = nn.MSELoss()  # Mean Squared Error for Continuous Targets
scaler = torch.amp.GradScaler('cuda')

config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/136M [00:00<?, ?B/s]

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_1407/3517066919.py:25: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler('cuda')


In [ ]:
print("\n--- Starting ESM-2 35M Continuous Fitness Regression ---")

for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        fitness_targets = batch["fitness"].to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            predictions = model(input_ids, attention_mask)
            loss = criterion(predictions, fitness_targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    # Evaluation
    model.eval()
    all_preds_continuous, all_targets_continuous = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            fitness_targets = batch["fitness"].to(device)

            with torch.amp.autocast('cuda'):
                preds = model(input_ids, attention_mask)

            all_preds_continuous.extend(preds.cpu().numpy())
            all_targets_continuous.extend(fitness_targets.cpu().numpy())

    preds_arr = np.array(all_preds_continuous)
    targets_arr = np.array(all_targets_continuous)

    # Ground truth binary labels for metric thresholding
    binary_targets = (targets_arr >= 0.5).astype(int)

    # Search threshold on continuous predictions to maximize MCC
    best_mcc, best_thresh, best_binary_preds = -1.0, 0.5, []
    for thresh in np.arange(0.2, 0.8, 0.02):
        b_preds = (preds_arr >= thresh).astype(int)
        mcc_val = matthews_corrcoef(binary_targets, b_preds)
        if mcc_val > best_mcc:
            best_mcc = mcc_val
            best_thresh = thresh
            best_binary_preds = b_preds

    acc = accuracy_score(binary_targets, best_binary_preds)
    f1 = f1_score(binary_targets, best_binary_preds, zero_division=0)
    r2 = r2_score(targets_arr, preds_arr)

    mse_val = total_loss / len(train_loader)
    print(f"Epoch {epoch:02d} | Train MSE: {mse_val:.4f} | R2: {r2:.4f} | Val Acc: {acc:.4f} | MCC: {best_mcc:.4f} | F1: {f1:.4f} | Optimal Thresh: {best_thresh:.2f}")

In [ ]:
# Ensure model is in evaluation mode
model.eval()

all_preds_continuous = []
all_targets_continuous = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        fitness_targets = batch["fitness"].to(device)

        with torch.amp.autocast('cuda'):
            preds = model(input_ids, attention_mask)

        all_preds_continuous.extend(preds.cpu().numpy())
        all_targets_continuous.extend(fitness_targets.cpu().numpy())

preds_arr = np.array(all_preds_continuous)
targets_arr = np.array(all_targets_continuous)

# Apply optimal threshold found at Epoch 15 (0.58)
OPTIMAL_THRESH = 0.58
binary_preds = (preds_arr >= OPTIMAL_THRESH).astype(int)
binary_targets = (targets_arr >= 0.5).astype(int)

# Create results DataFrame
results_df = pd.DataFrame({
    'sequence': test_df['sequence'].values,
    'true_fitness': targets_arr,
    'predicted_fitness': preds_arr,
    'true_label': binary_targets,
    'predicted_label': binary_preds,
    'correct_prediction': (binary_preds == binary_targets)
})

# Save to CSV
results_df.to_csv("esm2_tem1_test_predictions.csv", index=False)
print(f"Saved {len(results_df)} test predictions to 'esm2_tem1_test_predictions.csv'")

In [30]:
model.eval()

all_preds_continuous = []
all_targets_continuous = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        fitness_targets = batch["fitness"].to(device)

        with torch.amp.autocast('cuda'):
            preds = model(input_ids, attention_mask)

        all_preds_continuous.extend(preds.cpu().numpy())
        all_targets_continuous.extend(fitness_targets.cpu().numpy())

preds_arr = np.array(all_preds_continuous)
targets_arr = np.array(all_targets_continuous)

# Apply optimal threshold found at Epoch 15 (0.58)
OPTIMAL_THRESH = 0.58
binary_preds = (preds_arr >= OPTIMAL_THRESH).astype(int)
binary_targets = (targets_arr >= 0.5).astype(int)

# Create results DataFrame
results_df = pd.DataFrame({
    'sequence': test_df['sequence'].values,
    'true_fitness': targets_arr,
    'predicted_fitness': preds_arr,
    'true_label': binary_targets,
    'predicted_label': binary_preds,
    'correct_prediction': (binary_preds == binary_targets)
})

# Save to CSV
results_df.to_csv("esm2_tem1_test_predictions.csv", index=False)
print(f"Saved {len(results_df)} test predictions to 'esm2_tem1_test_predictions.csv'")

Saved 912 test predictions to 'esm2_tem1_test_predictions.csv'
